# 21 · E1: how hackable is each predictor → Table 1, threshold sweep, attacker-seed variance

For each predictor, the stored attackers perturb the BVCC test audio within the budget. An attack **succeeds** on an
utterance when the predictor's score rises by more than τ = 0.3 over the codec reconstruction x̂ (the "corrected
reference") while at least one independent measure (ESTOI, SI-SDR, WER, SpeechBERTScore) says the audio got worse.
The attack success rate (ASR) is the fraction of utterances where that happens.

No training: the fresh attackers of E1 and the five attacker seeds of the variance control are loaded from the
release; waveform PGD and random directions need no training. Recompute cost: 27 GPU jobs of about 15 minutes.

Table 1 also has the human-agreement columns from E2 (notebook 20); if E2 has not been recomputed, the released E2
outputs are used for those columns.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (can also be set through environment variables when run non-interactively)
# SOURCE = "recompute": re-run the evaluation from the released checkpoints (needs GPUs), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU only, seconds);
#                       this rewrites the committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch, account PAS2966), this kernel waits; "local": run in this kernel.
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1', 'e1_attacker_seeds']]
print(len(KEYS), "runs:", ", ".join(KEYS))

27 runs: e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e1_R3ms_s0, e1_R3ms_s1, e1_R3ms_s2, e1seed_R0_s0, e1seed_R3_s0, e1seed_R3ms_s0


In [4]:
if SOURCE == "recompute":
    # Idempotent: runs whose recomputed output already exists are skipped.
    repro.run_keys(KEYS, "recompute", RUN)
dirs = repro.run_analysis(SOURCE)
OUT, PAPER = dirs["out"], dirs["paper"]

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Which outputs the tables below were built from (recomputed from checkpoints vs original runs).
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}).value_counts()

released    27
Name: count, dtype: int64

## Table 1

In [6]:
print((PAPER / "tables" / "table1.tex").read_text())

% generated by analysis/paper_assets.py
\begin{tabular}{l c c c c c c}
\toprule
& offset & \multicolumn{3}{c}{ASR by attack space} & \multicolumn{2}{c}{human agreement} \\
\cmidrule(lr){3-5}\cmidrule(lr){6-7}
Predictor & $R(\hat{x})\!-\!R(x)$ & EnCodec & DAC & PGD & BVCC & OOD \\
\midrule
$R_0$ (no defense) & -0.09$\pm$0.02 & 0.226$\pm$0.205 & 0.065$\pm$0.113 & 0.999$\pm$0.002 & 0.922$\pm$0.004 & 0.731$\pm$0.006 \\
$R_1$ & -0.06$\pm$0.03 & 0.050$\pm$0.048 & 0.161$\pm$0.033 & 1.000$\pm$0.001 & 0.911$\pm$0.009 & 0.690$\pm$0.014 \\
$R_2$ & 0.00$\pm$0.07 & 0.017$\pm$0.006 & 0.141$\pm$0.025 & 1.000$\pm$0.000 & 0.918$\pm$0.003 & 0.682$\pm$0.008 \\
$R_3$ (EnCodec loop) & -0.27$\pm$0.25 & 0.137$\pm$0.179 & 0.127$\pm$0.031 & 1.000$\pm$0.000 & 0.920$\pm$0.001 & 0.637$\pm$0.046 \\
$R_3^{\mathrm{multi}}$ (multi-space) & 0.95$\pm$0.16 & 0.082$\pm$0.107 & 0.016$\pm$0.026 & 0.841$\pm$0.089 & 0.914$\pm$0.003 & 0.552$\pm$0.015 \\
noise pairs & -0.09$\pm$0.01 & 0.270$\pm$0.107 & 0.230$\pm$0.028 & 1.000$

## ASR under both references (raw input x as originally run, and the corrected x̂)

In [7]:
display(Markdown((OUT / "e1_both_refs.md").read_text()))

# E1 under both ceiling references

tau=0.3; panel margins {'estoi': 0.02, 'sisdr': 1.0, 'wer': 0.0, 'sbs': 0.02}; mean over seeds; mean-action candidate only.

## Predictor's own reconstruction offset R(x_hat)-R(x)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.072 |         -0.095 |          0 |           -0.095 |               -0.095 |
| R1              |      0.074 |         -0.064 |          0 |           -0.064 |               -0.064 |
| R2              |      0.081 |          0.001 |          0 |            0.001 |                0.001 |
| R3              |      0.081 |         -0.266 |          0 |           -0.266 |               -0.266 |
| R3ms            |      0.04  |          0.949 |          0 |            0.949 |                0.949 |
| baseline_noise  |      0.069 |         -0.085 |          0 |           -0.085 |               -0.085 |
| baseline_pgd    |      0.082 |         -0.077 |          0 |           -0.077 |               -0.077 |
| baseline_random |      0.09  |         -0.03  |          0 |           -0.03  |               -0.03  |

## ASR, ceiling referenced to raw input x (as run)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.12  |          0.181 |      0.999 |            0.012 |                0.251 |
| R1              |      0.301 |          0.067 |      1     |            0.001 |                0     |
| R2              |      0.304 |          0.079 |      1     |            0     |                0     |
| R3              |      0.293 |          0.05  |      1     |            0     |                0     |
| R3ms            |      0.037 |          0.306 |      0.841 |            0.022 |                0     |
| baseline_noise  |      0.356 |          0.235 |      1     |            0.013 |                0.217 |
| baseline_pgd    |      0.352 |          0.21  |      0.969 |            0.01  |                0.181 |
| baseline_random |      0.35  |          0.053 |      1     |            0     |                0.002 |

## ASR, ceiling referenced to codec reconstruction x_hat (corrected)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.065 |          0.226 |      0.999 |            0.001 |                0.318 |
| R1              |      0.161 |          0.05  |      1     |            0     |                0     |
| R2              |      0.141 |          0.017 |      1     |            0     |                0     |
| R3              |      0.127 |          0.137 |      1     |            0.002 |                0     |
| R3ms            |      0.016 |          0.082 |      0.841 |            0     |                0     |
| baseline_noise  |      0.23  |          0.27  |      1     |            0     |                0.246 |
| baseline_pgd    |      0.225 |          0.223 |      0.969 |            0     |                0.18  |
| baseline_random |      0.2   |          0.019 |      1     |            0     |                0     |

## Attacker gain over x_hat

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |     -0.051 |          0.136 |      1.786 |           -0.344 |                0.245 |
| R1              |      0.142 |          0.138 |      1.935 |           -0.913 |               -1.56  |
| R2              |      0.134 |          0.03  |      2.003 |           -1.303 |               -1.743 |
| R3              |      0.122 |          0.216 |      2.102 |           -1.614 |               -2.491 |
| R3ms            |      0.048 |         -0.455 |      1.227 |           -2.713 |               -3.51  |
| baseline_noise  |      0.171 |          0.223 |      2.04  |           -0.388 |                0.191 |
| baseline_pgd    |      0.176 |          0.205 |      1.618 |           -0.684 |                0.078 |
| baseline_random |      0.156 |          0.102 |      2.055 |           -1.887 |               -0.959 |


In [8]:
print((OUT / "e1_reference_check.md").read_text())

asr_ref_input: ASR(R0)-ASR(R3) per seed [0.143, 0.005, 0.247], mean +0.131, bootstrap 95% CI [+0.005, +0.247]
asr_ref_input: ASR(R0)-ASR(R3ms) per seed [0.083, -0.729, 0.272], mean -0.125, bootstrap 95% CI [-0.729, +0.272]
asr_ref_input: ASR(R0)-ASR(baseline_random) per seed [0.119, -0.017, 0.282], mean +0.128, bootstrap 95% CI [-0.017, +0.282]
asr_ref_recon: ASR(R0)-ASR(R3) per seed [-0.073, -0.039, 0.379], mean +0.089, bootstrap 95% CI [-0.073, +0.379]
asr_ref_recon: ASR(R0)-ASR(R3ms) per seed [0.227, -0.2, 0.404], mean +0.144, bootstrap 95% CI [-0.200, +0.404]
asr_ref_recon: ASR(R0)-ASR(baseline_random) per seed [0.234, -0.007, 0.394], mean +0.207, bootstrap 95% CI [-0.007, +0.394]


## Is the ranking of defences an artifact of τ? (fresh EnCodec attacker)

In [9]:
display(Markdown((OUT / "e1_tau_sweep.md").read_text()))

# Corrected-reference ASR vs detection threshold (fresh EnCodec attacker, mean over seeds)

| target          |   0.1 |   0.2 |   0.3 |   0.5 |   0.75 |   1.0 |   median_gap |   p90_gap |
|:----------------|------:|------:|------:|------:|-------:|------:|-------------:|----------:|
| R0              | 0.481 | 0.358 | 0.226 | 0.069 |  0.005 | 0     |        0.12  |     0.359 |
| R1              | 0.185 | 0.118 | 0.05  | 0.006 |  0     | 0     |        0.135 |     0.323 |
| R2              | 0.112 | 0.051 | 0.017 | 0.002 |  0     | 0     |        0.021 |     0.203 |
| R3              | 0.244 | 0.183 | 0.137 | 0.098 |  0.068 | 0.045 |        0.125 |     0.651 |
| R3ms            | 0.182 | 0.126 | 0.082 | 0.037 |  0.01  | 0.003 |       -0.495 |     0.135 |
| baseline_noise  | 0.63  | 0.447 | 0.27  | 0.071 |  0.009 | 0     |        0.203 |     0.465 |
| baseline_pgd    | 0.573 | 0.404 | 0.223 | 0.047 |  0.004 | 0     |        0.198 |     0.432 |
| baseline_random | 0.117 | 0.061 | 0.019 | 0.001 |  0     | 0     |        0.104 |     0.271 |


## Attacker-seed control: five attackers against one fixed predictor (seed 0)

In [10]:
av = pd.read_csv(OUT / "e1_attacker_variance.csv")
av.pivot(index="attacker", columns="target", values="asr_ref_recon").round(3)

target,R0,R3,R3ms
attacker,,,
a1,0.008,0.000,0.038
a2,0.225,0.382,0.080
a3,0.261,0.218,0.057
a4,0.000,0.001,0.086
a5,0.013,0.333,0.045


In [11]:
av.groupby("target").asr_ref_recon.agg(["mean", "std", "min", "max"]).round(3)

,mean,std,min,max
target,,,,
R0,0.102,0.130,0.000,0.261
R3,0.187,0.180,0.000,0.382
R3ms,0.061,0.021,0.038,0.086


In [12]:
print("\n".join(l for l in (OUT / "stats.md").read_text().split("\n\n") if l.startswith("H1")))

H1: ASR(R0)-ASR(R3) per seed = [0.143, 0.005, 0.247], mean 0.131; supported = True (rule: mean ≥ 0.10 and all seeds > 0)
H1-transfer [grpo_dac]: ASR drop per seed [-0.274, 0.072, -0.317], mean -0.173; transfers = False
H1-transfer [pgd_wave]: ASR drop per seed [-0.003, 0.0, 0.0], mean -0.001; transfers = False
H1-transfer [random_encodec]: ASR drop per seed [0.006, 0.018, 0.012], mean 0.012; transfers = False
H1-transfer [transfer_r0_round1]: ASR drop per seed [0.213, 0.233, 0.308], mean 0.251; transfers = True


In [13]:
# Every number the paper quotes from this experiment: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('e1_', 'asr_', 'gain_', 'av_', 'h1_'))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; "
      f"identical: {(cmp.abs_diff == 0).sum()}; |diff| <= 0.01: {(cmp.abs_diff <= 0.01).sum()}")
cmp.head(25)

1625 numbers; max |diff| = 0.0000; identical: 1625; |diff| <= 0.01: 1625


,paper,reproduced,abs_diff
asr_R0_grpo_dac,0.120,0.120,0.0
asr_R0_grpo_dac_2dp,0.120,0.120,0.0
asr_R0_grpo_dac_sd,0.192,0.192,0.0
asr_R0_grpo_dac_seed0,0.015,0.015,0.0
asr_R0_grpo_dac_seed1,0.341,0.341,0.0
asr_R0_grpo_dac_seed2,0.004,0.004,0.0
asr_R0_grpo_encodec,0.181,0.181,0.0
asr_R0_grpo_encodec_2dp,0.180,0.180,0.0
asr_R0_grpo_encodec_sd,0.140,0.140,0.0
asr_R0_grpo_encodec_seed0,0.205,0.205,0.0
